# 支持向量机（Support Vector Machine, SVM）从零开始

## 学习目标

本笔记本将带你系统掌握支持向量机，涵盖以下内容：

1. **直观理解**：什么是SVM，为何有效
2. **底层数学**：超平面、间隔、对偶问题、KKT条件
3. **核技巧**：将非线性问题转化为线性问题
4. **软间隔SVM**：处理线性不可分数据
5. **从零实现**：用Python实现SVM（二次规划）
6. **实际应用**：使用scikit-learn解决真实问题

---

## 目录

- [第1章：SVM直觉理解](#ch1)
- [第2章：线性SVM的数学推导](#ch2)
- [第3章：对偶问题与拉格朗日乘子](#ch3)
- [第4章：核技巧与非线性SVM](#ch4)
- [第5章：软间隔SVM（C-SVM）](#ch5)
- [第6章：从零实现SVM](#ch6)
- [第7章：scikit-learn实践](#ch7)
- [第8章：调参指南与总结](#ch8)

In [ ]:
# 导入所有需要的库
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.optimize import minimize
from sklearn import datasets
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# 设置绘图风格
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

np.random.seed(42)
print("环境准备完毕！")

<a id='ch1'></a>
---
## 第1章：SVM直觉理解

### 1.1 分类问题回顾

假设我们有两类数据点（正类 `+1` 和负类 `-1`），需要找到一条直线（或超平面）将它们分开。

**关键问题**：可能有无数条分割线，哪一条**最好**？

SVM的回答是：选择**间隔（margin）最大**的那条线。

> 间隔越大 → 对新数据的容错能力越强 → 泛化能力越好

In [ ]:
# ── 可视化：为什么间隔最大化更好 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 生成线性可分数据
np.random.seed(0)
X_pos = np.random.randn(10, 2) + np.array([2, 2])
X_neg = np.random.randn(10, 2) + np.array([-2, -2])
X_demo = np.vstack([X_pos, X_neg])
y_demo = np.array([1]*10 + [-1]*10)

x_range = np.linspace(-5, 5, 100)

for ax, title in zip(axes, ['任意分割线（间隔小）', 'SVM最优超平面（间隔最大）']):
    ax.scatter(X_pos[:, 0], X_pos[:, 1], c='royalblue', s=80,
               edgecolors='k', zorder=3, label='正类 (+1)')
    ax.scatter(X_neg[:, 0], X_neg[:, 1], c='tomato', s=80,
               marker='s', edgecolors='k', zorder=3, label='负类 (-1)')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_title(title, fontsize=13)
    ax.legend()

# 任意分割线（偏斜）
axes[0].plot(x_range, -0.3 * x_range + 0.5, 'k--', lw=2, label='分割线')
axes[0].annotate('间隔小\n→ 泛化差', xy=(1, 0.2), fontsize=11,
                 color='darkorange',
                 arrowprops=dict(arrowstyle='->', color='darkorange'),
                 xytext=(2.5, -2))

# SVM 最优超平面（近似）
axes[1].plot(x_range, -x_range, 'k-', lw=2.5, label='最优超平面')
axes[1].plot(x_range, -x_range + 2, 'b--', lw=1.5, alpha=0.7, label='间隔边界')
axes[1].plot(x_range, -x_range - 2, 'r--', lw=1.5, alpha=0.7)
axes[1].fill_between(x_range, -x_range - 2, -x_range + 2,
                     alpha=0.08, color='green', label='间隔区域')
axes[1].annotate('间隔大\n→ 泛化好', xy=(0, 0), fontsize=11,
                 color='green',
                 arrowprops=dict(arrowstyle='->', color='green'),
                 xytext=(2.5, -3))
axes[1].legend(fontsize=9)

plt.suptitle('SVM核心思想：最大化分类间隔', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 1.2 核心概念词汇表

| 术语 | 含义 |
|------|------|
| **超平面 (Hyperplane)** | n维空间中的 (n-1) 维子空间，即分类边界 |
| **间隔 (Margin)** | 两个类别离超平面最近点之间的距离 |
| **支持向量 (Support Vectors)** | 距离超平面最近的数据点，决定超平面位置 |
| **硬间隔 (Hard Margin)** | 数据完全线性可分时使用，不允许误分类 |
| **软间隔 (Soft Margin)** | 允许少量误分类，引入松弛变量 |
| **核函数 (Kernel)** | 将低维特征映射到高维，处理非线性问题 |

<a id='ch2'></a>
---
## 第2章：线性SVM的数学推导

### 2.1 超平面的数学表达

在 $n$ 维空间中，超平面定义为：

$$\mathbf{w}^T \mathbf{x} + b = 0$$

- $\mathbf{w} \in \mathbb{R}^n$：法向量（决定超平面方向）
- $b \in \mathbb{R}$：偏置项（决定超平面位置）
- $\mathbf{x} \in \mathbb{R}^n$：输入特征向量

**分类规则**：
$$\hat{y} = \text{sign}(\mathbf{w}^T \mathbf{x} + b)$$

### 2.2 点到超平面的距离

点 $\mathbf{x}_i$ 到超平面 $\mathbf{w}^T \mathbf{x} + b = 0$ 的**有符号距离**为：

$$d_i = \frac{\mathbf{w}^T \mathbf{x}_i + b}{\|\mathbf{w}\|}$$

**几何间隔** = 正类最近点距离 + 负类最近点距离 = $\dfrac{2}{\|\mathbf{w}\|}$

### 2.3 SVM原始优化问题

**目标**：最大化间隔，等价于最小化 $\|\mathbf{w}\|$

**硬间隔SVM** 的优化问题：

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2$$

$$\text{s.t.} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1, \quad i = 1, \ldots, m$$

> 约束条件含义：每个训练样本到超平面的距离 $\geq \dfrac{1}{\|\mathbf{w}\|}$，即正确分类且在间隔边界外。

等号成立的样本就是**支持向量**。

In [ ]:
# ── 可视化：间隔的几何意义 ──
fig, ax = plt.subplots(figsize=(8, 7))

# 数据点
X_p = np.array([[1.5, 3.0], [2.0, 4.5], [2.5, 2.5], [3.5, 4.0]])
X_n = np.array([[-1.0, 1.0], [-2.0, 3.0], [-1.5, 4.5], [-3.0, 2.0]])

ax.scatter(X_p[:, 0], X_p[:, 1], c='royalblue', s=120,
           edgecolors='k', zorder=5, label='正类 y=+1')
ax.scatter(X_n[:, 0], X_n[:, 1], c='tomato', s=120, marker='s',
           edgecolors='k', zorder=5, label='负类 y=-1')

# 超平面（w·x + b = 0）：x2 = -x1 + 1
xx = np.linspace(-4, 5, 200)
ax.plot(xx, -xx + 1.0, 'k-', lw=2.5, label='决策超平面: $\\mathbf{w}^T\\mathbf{x}+b=0$')
ax.plot(xx, -xx + 3.5, 'b--', lw=2, label='正类间隔边界: $\\mathbf{w}^T\\mathbf{x}+b=+1$')
ax.plot(xx, -xx - 1.5, 'r--', lw=2, label='负类间隔边界: $\\mathbf{w}^T\\mathbf{x}+b=-1$')
ax.fill_between(xx, -xx - 1.5, -xx + 3.5, alpha=0.08, color='green')

# 标注支持向量
sv_pos = np.array([1.5, 3.0])   # 近似支持向量
sv_neg = np.array([-1.0, 1.0])  # 近似支持向量
ax.scatter(*sv_pos, c='royalblue', s=300, edgecolors='gold',
           linewidths=3, zorder=6)
ax.scatter(*sv_neg, c='tomato', s=300, marker='s', edgecolors='gold',
           linewidths=3, zorder=6)

# 标注间隔
ax.annotate('', xy=(0.9, 2.6), xytext=(0.3, 1.2),
            arrowprops=dict(arrowstyle='<->', color='darkgreen', lw=2))
ax.text(0.1, 1.9, 'margin = $\\frac{2}{\\|\\mathbf{w}\\|}$',
        fontsize=12, color='darkgreen',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# 标注支持向量标签
ax.annotate('支持向量', xy=sv_pos, xytext=(3.2, 2.0),
            fontsize=11, color='navy',
            arrowprops=dict(arrowstyle='->', color='navy'))
ax.annotate('支持向量', xy=sv_neg, xytext=(-3.5, 0.0),
            fontsize=11, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

ax.set_xlim(-4.5, 5)
ax.set_ylim(-0.5, 6)
ax.set_xlabel('$x_1$', fontsize=13)
ax.set_ylabel('$x_2$', fontsize=13)
ax.set_title('SVM几何间隔示意图', fontsize=15)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

<a id='ch3'></a>
---
## 第3章：对偶问题与拉格朗日乘子

### 3.1 为什么要转化为对偶问题？

原始问题（Primal）直接求解 $\mathbf{w}, b$ 需要面对不等式约束，处理起来较复杂。  
通过**拉格朗日对偶**转化，可以：
- 将约束融入目标函数
- 方便引入核函数
- 利用已成熟的二次规划求解器

### 3.2 拉格朗日函数

引入拉格朗日乘子 $\alpha_i \geq 0$，构造：

$$L(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^{m} \alpha_i \left[ y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1 \right]$$

### 3.3 KKT条件

最优解必须满足 **KKT（Karush-Kuhn-Tucker）条件**：

| 条件 | 表达式 | 含义 |
|------|--------|------|
| 稳定性 | $\frac{\partial L}{\partial \mathbf{w}} = 0$ | $\mathbf{w} = \sum_{i} \alpha_i y_i \mathbf{x}_i$ |
| 稳定性 | $\frac{\partial L}{\partial b} = 0$ | $\sum_{i} \alpha_i y_i = 0$ |
| 原始可行性 | $y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1$ | 所有样本正确分类 |
| 对偶可行性 | $\alpha_i \geq 0$ | 乘子非负 |
| 互补松弛 | $\alpha_i[y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1] = 0$ | 非支持向量的 $\alpha_i = 0$ |

**互补松弛条件的深刻含义**：
- 若 $\alpha_i > 0$，则 $y_i(\mathbf{w}^T \mathbf{x}_i + b) = 1$（即 $\mathbf{x}_i$ 是支持向量）
- 若 $y_i(\mathbf{w}^T \mathbf{x}_i + b) > 1$（非支持向量），则 $\alpha_i = 0$

### 3.4 对偶问题（Wolfe Dual）

将 KKT 稳定性条件代回 $L$，得到**对偶目标函数**：

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{m} \alpha_i - \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j$$

$$\text{s.t.} \quad \alpha_i \geq 0, \quad \sum_{i=1}^{m} \alpha_i y_i = 0$$

求解得到 $\boldsymbol{\alpha}^*$ 后，可恢复：
$$\mathbf{w}^* = \sum_{i} \alpha_i^* y_i \mathbf{x}_i$$
$$b^* = y_j - \mathbf{w}^{*T} \mathbf{x}_j \quad (\text{对任意支持向量} \mathbf{x}_j)$$

**预测**：
$$\hat{y} = \text{sign}\left(\sum_{i} \alpha_i^* y_i \mathbf{x}_i^T \mathbf{x} + b^*\right)$$

> **关键观察**：预测只需计算 $\mathbf{x}_i^T \mathbf{x}$（内积），这为引入**核技巧**奠定基础！

In [ ]:
# ── 演示：用scipy求解对偶问题（小规模手动实现）──

def solve_svm_dual(X, y):
    """
    用scipy.optimize求解SVM对偶问题（硬间隔）
    最大化: sum(alpha) - 0.5 * alpha^T @ Q @ alpha
    其中 Q[i,j] = y[i]*y[j]*x[i]·x[j]
    约束: alpha >= 0, sum(alpha * y) = 0
    """
    m = len(y)
    # Gram矩阵
    Q = np.outer(y, y) * (X @ X.T)

    # 目标函数（取负，因为scipy做最小化）
    def objective(alpha):
        return 0.5 * alpha @ Q @ alpha - np.sum(alpha)

    def grad(alpha):
        return Q @ alpha - np.ones(m)

    # 约束: sum(alpha * y) = 0
    constraints = {'type': 'eq', 'fun': lambda a: np.dot(a, y),
                   'jac': lambda a: y}
    # 边界: alpha >= 0
    bounds = [(0, None)] * m

    result = minimize(objective, np.zeros(m), jac=grad,
                      method='SLSQP', bounds=bounds,
                      constraints=constraints,
                      options={'ftol': 1e-9, 'maxiter': 1000})

    alpha = result.x
    # 提取支持向量（alpha > threshold）
    sv_mask = alpha > 1e-5
    w = (alpha * y) @ X
    # 用支持向量求 b（取均值以提高数值稳定性）
    b = np.mean(y[sv_mask] - X[sv_mask] @ w)

    return w, b, alpha, sv_mask


# 生成线性可分数据
np.random.seed(7)
X_lin = np.r_[np.random.randn(12, 2) + [2, 2],
              np.random.randn(12, 2) - [2, 2]]
y_lin = np.array([1]*12 + [-1]*12)

w_sol, b_sol, alphas, sv_idx = solve_svm_dual(X_lin, y_lin)

print("=== 对偶问题求解结果 ===")
print(f"法向量 w = {w_sol.round(4)}")
print(f"偏置   b = {b_sol:.4f}")
print(f"支持向量数量: {sv_idx.sum()}")
print(f"间隔大小 (2/||w||): {2/np.linalg.norm(w_sol):.4f}")
print(f"支持向量索引: {np.where(sv_idx)[0]}")
print("\n各样本的 alpha 值（非零的是支持向量）:")
for i, a in enumerate(alphas):
    if a > 1e-5:
        print(f"  样本{i:2d}: alpha={a:.4f}, y={y_lin[i]:+d}, x={X_lin[i].round(3)}")

In [ ]:
# ── 可视化：支持向量与决策边界 ──
def plot_svm_boundary(X, y, w, b, alphas, sv_mask, title='SVM决策边界与支持向量'):
    fig, ax = plt.subplots(figsize=(8, 6))

    # 绘制数据点
    for label, color, marker in [(1, 'royalblue', 'o'), (-1, 'tomato', 's')]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], c=color, s=80,
                   marker=marker, edgecolors='k', zorder=3,
                   label=f'y={label:+d}')

    # 高亮支持向量
    ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=250,
               facecolors='none', edgecolors='gold',
               linewidths=3, zorder=4, label='支持向量')

    # 超平面与间隔边界
    x1_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300)
    if abs(w[1]) > 1e-10:
        hyperplane  = (-w[0] * x1_range - b) / w[1]
        margin_pos  = (-w[0] * x1_range - b + 1) / w[1]
        margin_neg  = (-w[0] * x1_range - b - 1) / w[1]
        ax.plot(x1_range, hyperplane, 'k-', lw=2.5, label='决策超平面')
        ax.plot(x1_range, margin_pos, 'b--', lw=1.8, alpha=0.8, label='间隔边界 (+1/-1)')
        ax.plot(x1_range, margin_neg, 'r--', lw=1.8, alpha=0.8)
        ax.fill_between(x1_range, margin_neg, margin_pos,
                        alpha=0.08, color='green', label='间隔区域')

    margin_width = 2 / np.linalg.norm(w)
    ax.set_title(f'{title}\n间隔 = {margin_width:.3f}，支持向量数 = {sv_mask.sum()}',
                 fontsize=13)
    ax.set_xlabel('$x_1$', fontsize=12)
    ax.set_ylabel('$x_2$', fontsize=12)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

plot_svm_boundary(X_lin, y_lin, w_sol, b_sol, alphas, sv_idx)

<a id='ch4'></a>
---
## 第4章：核技巧与非线性SVM

### 4.1 问题：线性不可分

当数据在原始特征空间中线性不可分时，SVM的硬间隔失效。  
解决思路：**将数据映射到更高维空间**，在高维空间中线性可分。

$$\phi: \mathbb{R}^n \rightarrow \mathbb{R}^N \quad (N \gg n)$$

### 4.2 核函数的妙处

直接计算高维映射 $\phi(\mathbf{x})$ 开销巨大（甚至无穷维），但对偶问题中只需要**内积**：

$$\mathbf{x}_i^T \mathbf{x}_j \rightarrow \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j) = K(\mathbf{x}_i, \mathbf{x}_j)$$

只要定义核函数 $K$，无需显式计算 $\phi$！这就是**核技巧（Kernel Trick）**。

### 4.3 常用核函数

| 核函数 | 公式 | 特点 |
|--------|------|------|
| **线性核** | $K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T \mathbf{x}_j$ | 线性可分，速度最快 |
| **多项式核** | $K(\mathbf{x}_i, \mathbf{x}_j) = (\gamma\, \mathbf{x}_i^T \mathbf{x}_j + r)^d$ | 捕捉多项式特征交互 |
| **RBF/高斯核** | $K(\mathbf{x}_i, \mathbf{x}_j) = e^{-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2}$ | 最常用，可处理任意非线性 |
| **Sigmoid核** | $K(\mathbf{x}_i, \mathbf{x}_j) = \tanh(\gamma\, \mathbf{x}_i^T \mathbf{x}_j + r)$ | 类似神经网络激活函数 |

### 4.4 Mercer定理

函数 $K(\mathbf{x}, \mathbf{z})$ 是合法核函数的充要条件：**对应的核矩阵（Gram矩阵）是半正定的**。

$$K_{ij} = K(\mathbf{x}_i, \mathbf{x}_j) \succeq 0$$

In [ ]:
# ── 核技巧可视化：低维不可分 → 高维可分 ──

# 生成同心圆数据
np.random.seed(42)
n = 100
r1 = np.random.uniform(0, 1.5, n // 2)
r2 = np.random.uniform(2.5, 4.0, n // 2)
theta = np.random.uniform(0, 2 * np.pi, n // 2)

X_inner = np.c_[r1 * np.cos(theta), r1 * np.sin(theta)] + np.random.randn(n//2, 2)*0.1
X_outer = np.c_[r2 * np.cos(theta), r2 * np.sin(theta)] + np.random.randn(n//2, 2)*0.1
X_circle = np.vstack([X_inner, X_outer])
y_circle = np.array([1] * (n//2) + [-1] * (n//2))

# 特征映射: (x1, x2) → (x1, x2, x1^2 + x2^2)
def rbf_feature_map(X):
    return np.c_[X, X[:, 0]**2 + X[:, 1]**2]

X_mapped = rbf_feature_map(X_circle)

fig = plt.figure(figsize=(14, 5))

# 原始二维空间
ax1 = fig.add_subplot(121)
for label, color, marker in [(1,'royalblue','o'), (-1,'tomato','s')]:
    mask = y_circle == label
    ax1.scatter(X_circle[mask, 0], X_circle[mask, 1],
                c=color, s=40, marker=marker, edgecolors='k', alpha=0.7)
ax1.set_title('原始空间（线性不可分）', fontsize=13)
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
circle_inner = plt.Circle((0, 0), 1.5, fill=False, color='blue', linestyle='--', lw=2)
circle_outer = plt.Circle((0, 0), 2.5, fill=False, color='red', linestyle='--', lw=2)
ax1.add_patch(circle_inner)
ax1.add_patch(circle_outer)
ax1.set_aspect('equal')

# 映射后三维空间
ax2 = fig.add_subplot(122, projection='3d')
for label, color in [(1, 'royalblue'), (-1, 'tomato')]:
    mask = y_circle == label
    ax2.scatter(X_mapped[mask, 0], X_mapped[mask, 1], X_mapped[mask, 2],
                c=color, s=40, alpha=0.7,
                edgecolors='k', linewidths=0.3)

# 绘制分割超平面（z = 常数）
xx, yy = np.meshgrid(np.linspace(-4, 4, 20), np.linspace(-4, 4, 20))
zz = np.ones_like(xx) * 6.5  # 超平面 z = 6.5
ax2.plot_surface(xx, yy, zz, alpha=0.2, color='green')

ax2.set_title('映射后空间（线性可分！）', fontsize=13)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_1^2 + x_2^2$')

blue_patch = mpatches.Patch(color='royalblue', label='内圈 y=+1')
red_patch = mpatches.Patch(color='tomato', label='外圈 y=-1')
ax2.legend(handles=[blue_patch, red_patch], loc='upper left')

plt.suptitle('核技巧：低维不可分 → 高维可分', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── 可视化：不同核函数的决策边界对比 ──
X_moons, y_moons = datasets.make_moons(n_samples=150, noise=0.15, random_state=42)

kernels = [
    ('linear',  {'kernel': 'linear', 'C': 1.0}, '线性核 (Linear)'),
    ('poly',    {'kernel': 'poly',  'C': 1.0, 'degree': 3, 'gamma': 'auto'}, '多项式核 (d=3)'),
    ('rbf',     {'kernel': 'rbf',   'C': 1.0, 'gamma': 'scale'}, 'RBF核（高斯核）'),
    ('sigmoid', {'kernel': 'sigmoid','C': 1.0, 'gamma': 'scale'}, 'Sigmoid核'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
xx, yy = np.meshgrid(np.linspace(-2.5, 3.5, 300),
                     np.linspace(-1.5, 2.5, 300))
cmap_bg = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_pt = ListedColormap(['tomato', 'royalblue'])

for ax, (name, params, title) in zip(axes, kernels):
    clf = SVC(**params)
    clf.fit(X_moons, y_moons)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_bg)
    ax.contour(xx, yy, Z, colors='k', linewidths=1.5)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons,
               cmap=cmap_pt, s=40, edgecolors='k', linewidths=0.5)
    # 支持向量
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='gold', linewidths=2.5)
    acc = clf.score(X_moons, y_moons)
    n_sv = len(clf.support_vectors_)
    ax.set_title(f'{title}\nAcc={acc:.2%}, SV={n_sv}', fontsize=11)
    ax.set_xlim(-2.5, 3.5)
    ax.set_ylim(-1.5, 2.5)

plt.suptitle('不同核函数对月牙形数据的决策边界', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<a id='ch5'></a>
---
## 第5章：软间隔SVM（C-SVM）

### 5.1 为什么需要软间隔？

实际数据通常存在**噪声和重叠**，硬间隔SVM会：
- 找不到可行解（完全不可分）
- 过拟合（间隔过窄）

**解决方案**：引入**松弛变量** $\xi_i \geq 0$，允许样本违反间隔约束。

### 5.2 软间隔优化问题

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{m} \xi_i$$

$$\text{s.t.} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0, \quad i = 1, \ldots, m$$

**$C$ 的作用（正则化参数）**：

| $C$ 值 | 效果 | 风险 |
|--------|------|------|
| $C \rightarrow \infty$ | 硬间隔，不允许误分类 | 过拟合 |
| $C$ 大 | 间隔小，支持向量少，对误分类惩罚重 | 过拟合 |
| $C$ 小 | 间隔大，支持向量多，对误分类容忍 | 欠拟合 |

### 5.3 软间隔对偶问题

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{m} \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j)$$

$$\text{s.t.} \quad 0 \leq \alpha_i \leq C, \quad \sum_{i=1}^{m} \alpha_i y_i = 0$$

与硬间隔相比，$\alpha_i$ 多了上界 $C$（box constraint）。

### 5.4 Hinge Loss 视角

软间隔SVM等价于最小化**Hinge Loss + L2正则化**：

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{m} \max(0, 1 - y_i(\mathbf{w}^T \mathbf{x}_i + b))$$

Hinge Loss: $\ell(y, f) = \max(0, 1 - y \cdot f)$

In [ ]:
# ── 可视化：Hinge Loss 与其他损失函数对比 ──
z = np.linspace(-2.5, 3, 500)  # z = y*f (函数间隔)

hinge    = np.maximum(0, 1 - z)
logistic = np.log1p(np.exp(-z))
square   = (1 - z)**2
zero_one = (z < 0).astype(float)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(z, hinge,    'b-',  lw=2.5, label='Hinge Loss (SVM)')
ax.plot(z, logistic, 'g--', lw=2.5, label='Logistic Loss (LR)')
ax.plot(z, square,   'r:',  lw=2.5, label='Squared Loss')
ax.step(z, zero_one, 'k-.', lw=2,   label='0-1 Loss (理想)')

ax.axvline(x=1, color='gray', linestyle=':', alpha=0.7)
ax.axvline(x=0, color='gray', linestyle=':',  alpha=0.7)
ax.axhline(y=0, color='black', lw=0.8)
ax.fill_between(z, 0, hinge, where=(z < 1), alpha=0.08, color='blue',
                label='Hinge Loss > 0 的区域')

ax.annotate('正确分类\n且在间隔外', xy=(2.0, 0), xytext=(1.8, 0.8),
            fontsize=10, color='darkblue',
            arrowprops=dict(arrowstyle='->', color='darkblue'))
ax.annotate('间隔内\n或误分类', xy=(0.5, 0.5), xytext=(-1, 1.5),
            fontsize=10, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

ax.set_xlabel('函数间隔 $z = y \\cdot f(\\mathbf{x})$', fontsize=12)
ax.set_ylabel('损失值', fontsize=12)
ax.set_title('Hinge Loss 与常见损失函数对比', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(-0.2, 3)
ax.set_xlim(-2.5, 3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 可视化：C 参数对决策边界的影响 ──
X_noisy, y_noisy = datasets.make_classification(
    n_samples=120, n_features=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=0.8, random_state=42)
y_noisy = np.where(y_noisy == 0, -1, 1)

C_values = [0.01, 0.1, 1.0, 100.0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
xx, yy = np.meshgrid(
    np.linspace(X_noisy[:, 0].min()-0.5, X_noisy[:, 0].max()+0.5, 300),
    np.linspace(X_noisy[:, 1].min()-0.5, X_noisy[:, 1].max()+0.5, 300))

for ax, C in zip(axes, C_values):
    clf = SVC(kernel='rbf', C=C, gamma='scale')
    clf.fit(X_noisy, y_noisy)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_bg)
    ax.contour(xx, yy, Z, colors='k', linewidths=1.5)
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1],
               c=np.where(y_noisy==1, 1, 0),
               cmap=cmap_pt, s=40, edgecolors='k', linewidths=0.5)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='gold', linewidths=2.5)
    acc = clf.score(X_noisy, y_noisy)
    n_sv = len(clf.support_vectors_)
    ax.set_title(f'C = {C}\nAcc={acc:.2%}, SV={n_sv}', fontsize=11)

plt.suptitle('正则化参数 C 对 SVM 的影响（RBF核）', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("C越小：间隔越大，容忍更多误分类（欠拟合倾向）")
print("C越大：间隔越小，减少误分类（过拟合倾向）")

<a id='ch6'></a>
---
## 第6章：从零实现SVM

### 6.1 核SVM的完整实现

我们将实现一个支持多种核函数的软间隔SVM，使用二次规划求解。

In [ ]:
class SVMFromScratch:
    """
    从零实现的支持向量机（软间隔 + 核技巧）

    使用 scipy.optimize.minimize 求解对偶二次规划问题：
        max  Σ αᵢ - ½ Σᵢ Σⱼ αᵢ αⱼ yᵢ yⱼ K(xᵢ, xⱼ)
        s.t. 0 ≤ αᵢ ≤ C
             Σ αᵢ yᵢ = 0

    Parameters
    ----------
    C      : float  正则化参数
    kernel : str    核函数类型 ('linear', 'rbf', 'poly')
    gamma  : float  RBF/poly 核参数
    degree : int    多项式核次数
    tol    : float  支持向量判别阈值
    """

    def __init__(self, C=1.0, kernel='rbf', gamma=1.0, degree=3, tol=1e-5):
        self.C = C
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.tol = tol

    # ── 核函数 ──────────────────────────────────────────────
    def _kernel(self, X1, X2):
        if self.kernel == 'linear':
            return X1 @ X2.T
        elif self.kernel == 'rbf':
            # K(x,z) = exp(-gamma * ||x-z||^2)
            # 利用 ||x-z||^2 = ||x||^2 + ||z||^2 - 2x·z
            sq_dist = (np.sum(X1**2, axis=1, keepdims=True)
                       + np.sum(X2**2, axis=1)
                       - 2 * X1 @ X2.T)
            return np.exp(-self.gamma * sq_dist)
        elif self.kernel == 'poly':
            return (self.gamma * X1 @ X2.T + 1) ** self.degree
        else:
            raise ValueError(f"未知核函数: {self.kernel}")

    # ── 训练 ────────────────────────────────────────────────
    def fit(self, X, y):
        """
        X: (m, n) 训练特征
        y: (m,)   标签 {+1, -1}
        """
        m, n = X.shape
        self.X_train = X.copy()
        self.y_train = y.copy()

        # 计算核矩阵
        K = self._kernel(X, X)  # (m, m)

        # Gram矩阵：Q[i,j] = yᵢ yⱼ K(xᵢ, xⱼ)
        Q = np.outer(y, y) * K

        # 目标函数（最小化 -对偶目标）
        def objective(alpha):
            return 0.5 * alpha @ Q @ alpha - np.sum(alpha)

        def grad_objective(alpha):
            return Q @ alpha - np.ones(m)

        # 约束：Σ αᵢ yᵢ = 0
        constraints = [
            {'type': 'eq',
             'fun':  lambda a: np.dot(a, y),
             'jac':  lambda a: y.astype(float)}
        ]

        # 边界：0 ≤ αᵢ ≤ C
        bounds = [(0, self.C)] * m

        # 求解
        result = minimize(
            objective, np.zeros(m), jac=grad_objective,
            method='SLSQP', bounds=bounds,
            constraints=constraints,
            options={'ftol': 1e-9, 'maxiter': 2000})

        self.alphas = result.x

        # 支持向量判别
        self.sv_mask = self.alphas > self.tol
        self.sv_X = X[self.sv_mask]
        self.sv_y = y[self.sv_mask]
        self.sv_alphas = self.alphas[self.sv_mask]

        # 计算偏置 b：用在间隔边界上的支持向量（0 < α < C）
        # 这些满足 yᵢ(Σⱼ αⱼ yⱼ K(xⱼ,xᵢ) + b) = 1
        margin_sv = (self.alphas > self.tol) & (self.alphas < self.C - self.tol)
        if margin_sv.sum() == 0:
            margin_sv = self.sv_mask  # fallback

        K_sv = self._kernel(self.sv_X, X[margin_sv])  # (n_sv, n_margin)
        decision_at_margin = (self.sv_alphas * self.sv_y) @ K_sv
        self.b = np.mean(y[margin_sv] - decision_at_margin)

        return self

    # ── 决策函数 ─────────────────────────────────────────────
    def decision_function(self, X):
        """返回原始决策值（带符号距离）"""
        K = self._kernel(self.sv_X, X)  # (n_sv, m_test)
        return (self.sv_alphas * self.sv_y) @ K + self.b

    # ── 预测 ─────────────────────────────────────────────────
    def predict(self, X):
        return np.sign(self.decision_function(X))

    # ── 准确率 ───────────────────────────────────────────────
    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print("SVMFromScratch 类定义完毕！")

In [ ]:
# ── 测试1：线性可分数据 ──
print("=" * 50)
print("测试1：线性可分数据（线性核）")
print("=" * 50)

svm_linear = SVMFromScratch(C=1e5, kernel='linear')
svm_linear.fit(X_lin, y_lin)

acc_train = svm_linear.score(X_lin, y_lin)
print(f"训练准确率: {acc_train:.4f}")
print(f"支持向量数: {svm_linear.sv_mask.sum()}")
print(f"决策函数值示例: {svm_linear.decision_function(X_lin[:3]).round(3)}")

# 可视化
fig, ax = plt.subplots(figsize=(7, 5))
xx1 = np.linspace(X_lin[:, 0].min()-1, X_lin[:, 0].max()+1, 300)
yy1 = np.linspace(X_lin[:, 1].min()-1, X_lin[:, 1].max()+1, 300)
XX, YY = np.meshgrid(xx1, yy1)
Z = svm_linear.predict(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
ax.contourf(XX, YY, Z, alpha=0.25, cmap=cmap_bg)
ax.contour(XX, YY, Z, colors='k', linewidths=2)
for label, color, marker in [(1,'royalblue','o'), (-1,'tomato','s')]:
    mask = y_lin == label
    ax.scatter(X_lin[mask,0], X_lin[mask,1], c=color, s=70,
               marker=marker, edgecolors='k')
ax.scatter(svm_linear.sv_X[:,0], svm_linear.sv_X[:,1],
           s=250, facecolors='none', edgecolors='gold', linewidths=3,
           label='支持向量')
ax.set_title(f'从零实现SVM（线性核）\n训练准确率: {acc_train:.2%}', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 测试2：非线性数据（RBF核）──
print("=" * 50)
print("测试2：月牙形数据（RBF核）")
print("=" * 50)

X_m, y_m = datasets.make_moons(n_samples=80, noise=0.1, random_state=0)
y_m = np.where(y_m == 0, -1, 1)

svm_rbf = SVMFromScratch(C=5.0, kernel='rbf', gamma=1.0)
svm_rbf.fit(X_m, y_m)

acc_rbf = svm_rbf.score(X_m, y_m)
print(f"训练准确率: {acc_rbf:.4f}")
print(f"支持向量数: {svm_rbf.sv_mask.sum()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, title in zip(axes,
    [svm_rbf, SVC(C=5.0, kernel='rbf', gamma=1.0)],
    ['从零实现 SVM（RBF核）', 'scikit-learn SVC（参照）']):
    if hasattr(model, 'fit') and not hasattr(model, 'alphas'):
        model.fit(X_m, y_m)
    xx2 = np.linspace(X_m[:, 0].min()-0.5, X_m[:, 0].max()+0.5, 300)
    yy2 = np.linspace(X_m[:, 1].min()-0.5, X_m[:, 1].max()+0.5, 300)
    XX2, YY2 = np.meshgrid(xx2, yy2)
    Z2 = model.predict(np.c_[XX2.ravel(), YY2.ravel()]).reshape(XX2.shape)
    ax.contourf(XX2, YY2, Z2, alpha=0.3, cmap=cmap_bg)
    ax.contour(XX2, YY2, Z2, colors='k', linewidths=1.5)
    ax.scatter(X_m[:, 0], X_m[:, 1],
               c=np.where(y_m==1, 1, 0),
               cmap=cmap_pt, s=60, edgecolors='k')
    acc = model.score(X_m, y_m)
    if hasattr(model, 'sv_X'):
        sv = model.sv_X
    else:
        sv = model.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1],
               s=250, facecolors='none', edgecolors='gold', linewidths=2.5)
    ax.set_title(f'{title}\nAcc={acc:.2%}, SV={len(sv)}', fontsize=12)

plt.suptitle('从零实现 vs scikit-learn 对比', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 测试3：多项式核 ──
print("=" * 50)
print("测试3：同心圆数据（多项式核 d=2）")
print("=" * 50)

svm_poly = SVMFromScratch(C=2.0, kernel='poly', gamma=1.0, degree=2)
svm_poly.fit(X_circle[:60], y_circle[:60])
acc_poly = svm_poly.score(X_circle[:60], y_circle[:60])
print(f"训练准确率: {acc_poly:.4f}")
print(f"支持向量数: {svm_poly.sv_mask.sum()}")

fig, ax = plt.subplots(figsize=(6, 6))
xx_c = np.linspace(-4.5, 4.5, 300)
yy_c = np.linspace(-4.5, 4.5, 300)
XXc, YYc = np.meshgrid(xx_c, yy_c)
Zc = svm_poly.predict(np.c_[XXc.ravel(), YYc.ravel()]).reshape(XXc.shape)
ax.contourf(XXc, YYc, Zc, alpha=0.25, cmap=cmap_bg)
ax.contour(XXc, YYc, Zc, colors='k', linewidths=2)
for label, color, marker in [(1,'royalblue','o'), (-1,'tomato','s')]:
    mask = y_circle[:60] == label
    ax.scatter(X_circle[:60][mask,0], X_circle[:60][mask,1],
               c=color, s=70, marker=marker, edgecolors='k')
ax.scatter(svm_poly.sv_X[:,0], svm_poly.sv_X[:,1],
           s=250, facecolors='none', edgecolors='gold', linewidths=3,
           label='支持向量')
ax.set_aspect('equal')
ax.set_title(f'从零实现 SVM（多项式核 d=2）\n训练准确率: {acc_poly:.2%}', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

<a id='ch7'></a>
---
## 第7章：scikit-learn实践

### 7.1 SVM用于真实数据集

下面使用**鸢尾花数据集（Iris）**和**乳腺癌数据集（Breast Cancer）**进行实践。

In [ ]:
# ── 鸢尾花数据集：多分类SVM（OvR / OvO）──
iris = datasets.load_iris()
X_iris, y_iris = iris.data, iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.25, random_state=42, stratify=y_iris)

pipe_iris = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(kernel='rbf', C=10, gamma='scale', decision_function_shape='ovr'))
])
pipe_iris.fit(X_train_iris, y_train_iris)
y_pred_iris = pipe_iris.predict(X_test_iris)

print("=== 鸢尾花数据集（全部4个特征）===")
print(f"训练集大小: {X_train_iris.shape}, 测试集大小: {X_test_iris.shape}")
print(f"测试准确率: {pipe_iris.score(X_test_iris, y_test_iris):.4f}")
print("\n分类报告:")
print(classification_report(y_test_iris, y_pred_iris,
                             target_names=iris.target_names))

In [ ]:
# ── 可视化：鸢尾花二维投影的决策区域（前两个特征）──
X_iris2 = X_iris[:, :2]  # 只用前两个特征用于可视化
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_iris2, y_iris, test_size=0.25, random_state=42, stratify=y_iris)

scaler2 = StandardScaler().fit(X_tr2)
X_tr2s, X_te2s = scaler2.transform(X_tr2), scaler2.transform(X_te2)

kernels_iris = [('linear', 1.0), ('rbf', 10.0), ('poly', 5.0)]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_iris = ['#FF6B6B', '#4ECDC4', '#45B7D1']
cmap3 = ListedColormap(colors_iris)

x_min, x_max = X_tr2s[:, 0].min()-0.5, X_tr2s[:, 0].max()+0.5
y_min, y_max = X_tr2s[:, 1].min()-0.5, X_tr2s[:, 1].max()+0.5
XXi, YYi = np.meshgrid(np.linspace(x_min, x_max, 300),
                        np.linspace(y_min, y_max, 300))

for ax, (k, C) in zip(axes, kernels_iris):
    clf = SVC(kernel=k, C=C, gamma='scale')
    clf.fit(X_tr2s, y_tr2)
    Zi = clf.predict(np.c_[XXi.ravel(), YYi.ravel()]).reshape(XXi.shape)
    ax.contourf(XXi, YYi, Zi, alpha=0.3, cmap=cmap3)
    ax.contour(XXi, YYi, Zi, colors='k', linewidths=1)
    ax.scatter(X_tr2s[:, 0], X_tr2s[:, 1], c=y_tr2, cmap=cmap3,
               s=50, edgecolors='k', linewidths=0.5, zorder=3)
    ax.scatter(X_te2s[:, 0], X_te2s[:, 1], c=y_te2, cmap=cmap3,
               s=80, marker='*', edgecolors='k', linewidths=0.8, zorder=4)
    # 支持向量
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=200, facecolors='none', edgecolors='gold', lw=2)
    acc = clf.score(X_te2s, y_te2)
    ax.set_title(f'核: {k}, C={C}\n测试准确率: {acc:.2%}', fontsize=12)
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

patches = [mpatches.Patch(color=c, label=n)
           for c, n in zip(colors_iris, iris.target_names)]
axes[1].legend(handles=patches, loc='upper right', fontsize=9)
plt.suptitle('鸢尾花数据集：不同核函数的SVM决策区域', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 乳腺癌数据集：二分类 + 混淆矩阵 ──
cancer = datasets.load_breast_cancer()
X_ca, y_ca = cancer.data, cancer.target

X_tr_ca, X_te_ca, y_tr_ca, y_te_ca = train_test_split(
    X_ca, y_ca, test_size=0.2, random_state=42, stratify=y_ca)

pipe_ca = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(kernel='rbf', C=10, gamma='scale', probability=True))
])
pipe_ca.fit(X_tr_ca, y_tr_ca)
y_pred_ca = pipe_ca.predict(X_te_ca)

print("=== 乳腺癌数据集 ===")
print(f"特征数: {X_ca.shape[1]}, 样本数: {X_ca.shape[0]}")
print(f"测试准确率: {pipe_ca.score(X_te_ca, y_te_ca):.4f}")
print("\n分类报告:")
print(classification_report(y_te_ca, y_pred_ca,
                             target_names=cancer.target_names))

# 混淆矩阵
cm = confusion_matrix(y_te_ca, y_pred_ca)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im)
tick_marks = np.arange(2)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(cancer.target_names, fontsize=11)
ax.set_yticklabels(cancer.target_names, fontsize=11)
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]}\n({cm[i,j]/cm[i].sum():.1%})',
                ha='center', va='center', fontsize=13,
                color='white' if cm[i, j] > thresh else 'black')
ax.set_ylabel('真实标签', fontsize=12)
ax.set_xlabel('预测标签', fontsize=12)
ax.set_title(f'混淆矩阵（RBF-SVM）\n准确率: {pipe_ca.score(X_te_ca, y_te_ca):.2%}',
             fontsize=13)
plt.tight_layout()
plt.show()

<a id='ch8'></a>
---
## 第8章：调参指南与总结

### 8.1 超参数调优（GridSearchCV）

In [ ]:
# ── GridSearchCV 调参示例 ──
param_grid = {
    'svc__C':     [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.1],
}

pipe_gs = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(kernel='rbf'))
])

gs = GridSearchCV(pipe_gs, param_grid, cv=5, scoring='accuracy',
                  n_jobs=-1, verbose=0)
gs.fit(X_tr_ca, y_tr_ca)

print("=== GridSearchCV 结果 ===")
print(f"最佳参数: {gs.best_params_}")
print(f"交叉验证最佳分数: {gs.best_score_:.4f}")
print(f"测试集准确率:    {gs.score(X_te_ca, y_te_ca):.4f}")

# 热力图可视化调参结果
results = gs.cv_results_
C_vals = [0.1, 1, 10, 100]
g_vals = ['scale', 'auto', 0.01, 0.1]
scores = gs.cv_results_['mean_test_score'].reshape(len(C_vals), len(g_vals))

fig, ax = plt.subplots(figsize=(8, 5))
im2 = ax.imshow(scores, interpolation='nearest', cmap='YlOrRd',
                vmin=scores.min(), vmax=scores.max())
plt.colorbar(im2, label='CV 准确率')
ax.set_xticks(range(len(g_vals)))
ax.set_yticks(range(len(C_vals)))
ax.set_xticklabels([str(g) for g in g_vals], fontsize=11)
ax.set_yticklabels([str(c) for c in C_vals], fontsize=11)
ax.set_xlabel('gamma', fontsize=12)
ax.set_ylabel('C', fontsize=12)
ax.set_title('GridSearchCV 超参数热力图（乳腺癌数据集）', fontsize=13)
for i in range(len(C_vals)):
    for j in range(len(g_vals)):
        ax.text(j, i, f'{scores[i,j]:.3f}', ha='center', va='center',
                fontsize=10,
                color='white' if scores[i,j] > scores.mean() else 'black')
plt.tight_layout()
plt.show()

### 8.2 SVM 调参实用指南

#### 第一步：选择核函数

```
特征数 >> 样本数？  → 线性核（Linear）
样本数 >> 特征数？  → RBF核（首选）
有领域先验知识？   → 多项式核 / 自定义核
```

#### 第二步：归一化数据

> **必须！** SVM对特征量纲敏感。使用 `StandardScaler` 将特征缩放到均值0，标准差1。

#### 第三步：调节 C 和 γ

推荐使用**指数网格搜索**：

```python
C_range     = [2**i for i in range(-5, 16, 2)]   # 2^{-5}, 2^{-3}, ..., 2^{15}
gamma_range = [2**i for i in range(-15, 4, 2)]   # 2^{-15}, ..., 2^{3}
```

#### 参数调整方向总结

| 问题 | 解决方向 |
|------|----------|
| 欠拟合（训练/测试准确率都低） | 增大 C，增大 γ（RBF） |
| 过拟合（训练高、测试低） | 减小 C，减小 γ |
| 训练太慢 | 减小 C，或使用线性核+LinearSVC |

### 8.3 SVM 优缺点总结

#### ✅ 优点
1. **高维效果好**：特征维度高于样本数时依然有效
2. **全局最优**：凸优化问题，无局部最优陷阱
3. **核技巧强大**：可处理各种非线性边界
4. **泛化理论扎实**：基于统计学习理论，有严格的泛化界
5. **内存高效**：预测只需支持向量

#### ❌ 缺点
1. **大规模数据慢**：训练复杂度 $O(m^2) \sim O(m^3)$
2. **调参敏感**：C、γ选择对性能影响大
3. **概率估计弱**：需要额外的Platt scaling（较慢）
4. **多分类需策略**：本质是二分类器，多分类需要OvO/OvR策略
5. **特征工程依赖**：核函数的选择需要一定经验

In [ ]:
# ── SVM 与其他分类器对比 ──
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

classifiers = [
    ('SVM (RBF)',        SVC(kernel='rbf', C=10, gamma='scale')),
    ('SVM (Linear)',     SVC(kernel='linear', C=1.0)),
    ('Logistic Reg.',    LogisticRegression(max_iter=1000)),
    ('Random Forest',    RandomForestClassifier(n_estimators=100, random_state=42)),
    ('KNN (k=5)',        KNeighborsClassifier(n_neighbors=5)),
    ('MLP (100,50)',     MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500,
                                      random_state=42)),
]

print(f"{'分类器':<22} {'训练准确率':>12} {'测试准确率':>12}")
print("-" * 48)

train_accs, test_accs, names = [], [], []
for name, clf in classifiers:
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_tr_ca, y_tr_ca)
    tr_acc = pipe.score(X_tr_ca, y_tr_ca)
    te_acc = pipe.score(X_te_ca, y_te_ca)
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    names.append(name)
    print(f"{name:<22} {tr_acc:>12.4f} {te_acc:>12.4f}")

# 条形图
x_pos = np.arange(len(names))
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x_pos - 0.2, train_accs, 0.38, label='训练准确率',
               color='steelblue', alpha=0.8, edgecolor='k')
bars2 = ax.bar(x_pos + 0.2, test_accs,  0.38, label='测试准确率',
               color='salmon', alpha=0.8, edgecolor='k')
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=10)
ax.set_ylim(0.88, 1.01)
ax.set_ylabel('准确率', fontsize=12)
ax.set_title('乳腺癌数据集：各分类器对比', fontsize=14)
ax.legend(fontsize=11)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

### 8.4 知识图谱总结

```
支持向量机 (SVM)
│
├── 线性SVM (硬间隔)
│   ├── 原始问题: min ½||w||²  s.t. yᵢ(w·xᵢ+b) ≥ 1
│   └── 对偶问题: max Σαᵢ - ½ΣΣαᵢαⱼyᵢyⱼxᵢ·xⱼ
│
├── 软间隔SVM (C-SVM)
│   ├── 引入松弛变量 ξᵢ
│   ├── C 控制间隔与误分类权衡
│   └── 等价 Hinge Loss + L2 正则化
│
├── 核SVM
│   ├── 核技巧: xᵢ·xⱼ → K(xᵢ,xⱼ)
│   ├── 线性核: xᵢᵀxⱼ
│   ├── RBF核: exp(-γ||xᵢ-xⱼ||²)  ← 最常用
│   └── 多项式核: (γxᵢᵀxⱼ+1)^d
│
├── 关键理论
│   ├── Mercer定理 (核函数合法性)
│   ├── KKT条件 (最优性条件)
│   └── VC维理论 (泛化界)
│
└── 实践要点
    ├── 必须归一化特征
    ├── 超参数: C, γ (指数搜索)
    └── 大数据 → LinearSVC / SGDClassifier
```

---

### 8.5 推荐学习资源

| 资源 | 类型 | 难度 |
|------|------|------|
| 《统计学习方法》- 李航 | 书籍 | ⭐⭐⭐ |
| CS229 SVM笔记 (Andrew Ng) | 讲义 | ⭐⭐ |
| LIBSVM - Chang & Lin (2011) | 论文 | ⭐⭐⭐ |
| 《Learning with Kernels》- Schölkopf | 书籍 | ⭐⭐⭐⭐ |
| scikit-learn SVM文档 | 文档 | ⭐ |

---

**🎉 恭喜你完成了SVM从零开始的学习！**

你现在已经掌握了：
- ✅ SVM的几何直觉
- ✅ 硬/软间隔的数学推导
- ✅ 拉格朗日对偶与KKT条件
- ✅ 核技巧的原理与常用核函数
- ✅ Python从零实现SVM
- ✅ scikit-learn实际应用与调参